# 06 — H6 arc semantics audit (human review)

Main-couple vs non-couple conflict, and topic × position weights \(W_{tkr}\).
High `4.4` label fidelity ≠ “is it **main-couple** conflict?”

Saved audits + packets only. Rating cells stay blinded; sentence tertiles are visible
because H6 is position-aware.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

cwd = Path.cwd().resolve()
root = cwd
for _ in range(6):
    if (root / "configs").is_dir() and (root / "src").is_dir():
        break
    root = root.parent
sys.path.insert(0, str(root))

from src.stage11_refined_construct_analysis.analysis import notebook_helpers as nh
from src.stage11_refined_construct_analysis.analysis import review_display as rd
from src.stage11_refined_construct_analysis.analysis.constructs import normalize_code

ctx = nh.setup("06_h6_arc_semantics_audit")
cfg = ctx.cfg
HYP = "H6"
CODE_COL = "arc_role"

Project root : /home/polina/Documents/Cursor_Projects/romantic_novels_large_corpus
Config       : configs/stage11/refined_constructs.yaml
Run          : v4_l12_granular_final_call49
Outputs      : results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/06_h6_arc_semantics_audit


## 1. Overview — arc roles with topic labels

In [2]:
master = nh.load_master(cfg)
h6 = master[master[CODE_COL].notna()].copy()
h6["code_norm"] = h6[CODE_COL].map(normalize_code)
print(f"H6-coded topics: {len(h6)}")

overview = rd.annotation_overview(
    h6,
    CODE_COL,
    extra_cols=["main_couple_prob", "non_couple_prob", "unclear_prob"],
)
display(
    overview[
        [
            "topic",
            "taxonomy",
            "code",
            "code_norm",
            "main_couple_prob",
            "non_couple_prob",
        ]
    ]
)
ctx.save_table(overview, "h6_topic_overview_labeled")
display(h6[CODE_COL].value_counts().head(25).to_frame("n"))
display(h6["code_norm"].value_counts(dropna=False).to_frame("n_norm"))

H6-coded topics: 54


,topic,taxonomy,code,code_norm,main_couple_prob,non_couple_prob
0,124 — Scooped Up in A Tight Hug,2.2 — Kissing & Non-Explicit Affection,ARC_7,ARC_7,0.8500,0.1000
1,126 — Fingertips Stroking Her Cheek,2.2 — Kissing & Non-Explicit Affection,ARC_7,ARC_7,0.8500,0.1000
2,102 — Grief Etched on His Face,3.2 — Negative Emotions & Distress,ARC_10,ARC_10,0.8500,0.1000
3,285 — Confessing Years of Hatred,3.2 — Negative Emotions & Distress,ARC_2,ARC_2,0.8500,0.1000
4,38 — Admitting Shared Pain,4.2 — Ongoing Courtship & Everyday Relational ...,ARC_10,ARC_10,0.8500,0.1000
5,232 — Conversation Cut Short By Arrival,4.2 — Ongoing Courtship & Everyday Relational ...,ARC_10,ARC_10,0.8500,0.1000
6,37 — Defending A Close Friendship,"4.3 — Secrets, Misunderstandings & Hidden Info...",ARC_1,ARC_1,0.8500,0.1000
7,94 — Caught in A Lie,"4.3 — Secrets, Misunderstandings & Hidden Info...",MIXED,None,0.8500,0.1000
8,109 — Seeing Past A Guarded Identity,"4.3 — Secrets, Misunderstandings & Hidden Info...",ARC_5,ARC_5,0.8500,0.1000
9,121 — Revealing Plans to The Prince,"4.3 — Secrets, Misunderstandings & Hidden Info...",ARC_0,ARC_0,0.1000,0.8000


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/06_h6_arc_semantics_audit/tables/h6_topic_overview_labeled.csv  (54 rows)


,n
arc_role,
ARC_7,10
ARC_0,9
ARC_1,7
ARC_10,7
MIXED,5
ARC_5,4
ARC_4,4
ARC_6,3
ARC_8,2


,n_norm
code_norm,
ARC_7,10
ARC_0,9
ARC_1,7
ARC_10,7
None,5
ARC_5,4
ARC_4,4
ARC_6,3
ARC_8,2


## 2. Lexical vs contextual agreement

In [3]:
lex = nh.load_audit_jsonl(cfg, HYP, "A")
ctxu = nh.load_audit_jsonl(cfg, HYP, "B")
adj = nh.load_audit_jsonl(cfg, HYP, "C")
lex_idx = rd.audit_index(lex)
ctx_idx = rd.audit_index(ctxu)
adj_idx = rd.audit_index(adj)

agree = rd.agreement_table(h6, lex_idx, ctx_idx, adj_idx, hyp=HYP)
if not agree.empty:
    print(
        f"Lexical–contextual agreement: {agree['agree'].mean():.1%} "
        f"({int(agree['agree'].sum())}/{len(agree)})"
    )
    ctx.save_table(agree, "h6_lexical_contextual_agreement")
    disagree = agree[~agree["agree"]]
    if len(disagree):
        display(
            disagree[
                ["topic", "taxonomy", "code_a", "code_b", "code_c", "rationale_c"]
            ]
        )

Lexical–contextual agreement: 35.2% (19/54)


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/06_h6_arc_semantics_audit/tables/h6_lexical_contextual_agreement.csv  (54 rows)


,topic,taxonomy,code_a,code_b,code_c,rationale_c
1,6 — Whispered Reassurance,"4.6 — Emotional Safety, Reassurance & Caretaking",ARC_10,ARC_0,ARC_0,The contextual dominant judgment (ARC_0) holds...
3,29 — Confessing Long-Held Love,"4.5 — Reconciliation, Commitments & HEA",MIXED,ARC_10,ARC_8,The lexical consensus (MIXED) and contextual d...
5,37 — Defending A Close Friendship,"4.3 — Secrets, Misunderstandings & Hidden Info...",ARC_0,ARC_1,ARC_1,Lexical consensus (ARC_0 / off_target) was ove...
6,38 — Admitting Shared Pain,4.2 — Ongoing Courtship & Everyday Relational ...,MIXED,ARC_10,ARC_10,The lexical consensus (MIXED) and contextual d...
7,43 — Pissed Off and Grumbling,"4.4 — Conflict, Distance & Breakup Threats",ARC_2,ARC_4,MIXED,Lexical consensus (ARC_2) and contextual domin...
8,45 — Reassured Everything Will Be Fine,"4.6 — Emotional Safety, Reassurance & Caretaking",ARC_7,ARC_10,ARC_7,The lexical consensus (ARC_7) and the taxonomy...
9,46 — Asking Someone to Trust You,"4.6 — Emotional Safety, Reassurance & Caretaking",ARC_1,ARC_7,ARC_7,"The taxonomy (4.6 Emotional Safety, Reassuranc..."
10,56 — Promising Never to Hurt You,"4.6 — Emotional Safety, Reassurance & Caretaking",ARC_4,ARC_6,ARC_6,The lexical consensus (ARC_4) reflects surface...
13,94 — Caught in A Lie,"4.3 — Secrets, Misunderstandings & Hidden Info...",ARC_5,ARC_1,MIXED,Lexical consensus (ARC_5 disclosure) and conte...
14,96 — Confessing Long-Standing Worry,"4.6 — Emotional Safety, Reassurance & Caretaking",ARC_10,ARC_4,ARC_4,The lexical consensus (ARC_10) reflects surfac...


## 3. Main-couple probabilities

Distribution first; then read low-probability topics that still carry conflict / secrecy
arc roles — those are the ones most likely to be external plot, not couple arc.

In [4]:
print(h6[["main_couple_prob", "non_couple_prob", "unclear_prob"]].describe())
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(h6["main_couple_prob"].dropna(), bins=20, color="steelblue", edgecolor="white")
ax.set_title("H6 main_couple_prob")
ctx.save_figure(fig, "h6_main_couple_hist")
plt.show()

low_mc = h6[h6["main_couple_prob"].fillna(0) < 0.4].copy()
print(f"Topics with main_couple_prob < 0.4: {len(low_mc)}")
if len(low_mc):
    display(
        rd.annotation_overview(low_mc, CODE_COL, extra_cols=["main_couple_prob"])[
            ["topic", "taxonomy", "code", "code_norm", "main_couple_prob"]
        ]
    )

       main_couple_prob  non_couple_prob  unclear_prob
count           54.0000          54.0000       54.0000
mean             0.7111           0.2296        0.0593
std              0.2941           0.2745        0.0196
min              0.1000           0.1000        0.0500
25%              0.8500           0.1000        0.0500
50%              0.8500           0.1000        0.0500
75%              0.8500           0.1000        0.0500
max              0.8500           0.8000        0.1000


  saved figure: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/06_h6_arc_semantics_audit/figures/h6_main_couple_hist.png
Topics with main_couple_prob < 0.4: 10


/tmp/ipykernel_99740/3171978054.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,topic,taxonomy,code,code_norm,main_couple_prob
0,121 — Revealing Plans to The Prince,"4.3 — Secrets, Misunderstandings & Hidden Info...",ARC_0,ARC_0,0.1000
1,130 — Revealing A Secret Plan,"4.3 — Secrets, Misunderstandings & Hidden Info...",ARC_0,ARC_0,0.1000
2,248 — Arranging A Cover Story,"4.3 — Secrets, Misunderstandings & Hidden Info...",ARC_10,ARC_10,0.1000
3,286 — Trying to Regain Good Graces,"4.3 — Secrets, Misunderstandings & Hidden Info...",ARC_0,ARC_0,0.1000
4,346 — Delivering Urgent News in Secret,"4.3 — Secrets, Misunderstandings & Hidden Info...",ARC_0,ARC_0,0.1000
5,256 — Refusing to Let It End,"4.4 — Conflict, Distance & Breakup Threats",ARC_0,ARC_0,0.1000
6,316 — Snapping Over Money and Control,"4.4 — Conflict, Distance & Breakup Threats",ARC_0,ARC_0,0.1000
7,6 — Whispered Reassurance,"4.6 — Emotional Safety, Reassurance & Caretaking",ARC_0,ARC_0,0.1000
8,36 — Eagerly Offering to Help,"4.6 — Emotional Safety, Reassurance & Caretaking",ARC_0,ARC_0,0.1000
9,338 — Promising Never to Do That Again,"9.2 — Promise, Vow & Future-Tense Speech Acts",ARC_0,ARC_0,0.1000


## 4. Topic × position weights \(W_{tkr}\)

In [5]:
wtkr = nh.load_w_tkr(cfg)
print(f"W_tkr rows: {len(wtkr)}")
if not wtkr.empty:
    wtkr = wtkr.copy()
    wtkr["code_norm"] = wtkr["construct_code"].map(normalize_code)
    display(
        wtkr.groupby(["tertile", "code_norm"], dropna=False)["weight"]
        .mean()
        .unstack(fill_value=0)
        .round(3)
    )
    ctx.save_table(wtkr, "w_tkr_raw")

W_tkr rows: 266


code_norm,ARC_0,ARC_1,ARC_10,ARC_2,ARC_3,ARC_4,ARC_5,ARC_6,ARC_7,ARC_8,ARC_9
tertile,,,,,,,,,,,
begin,0.7030,0.3690,0.6450,0.4180,0.5650,0.3560,0.5300,0.4300,0.7000,0.4950,0.5180
end,0.7000,0.4480,0.5970,0.4300,0.4470,0.4690,0.7200,0.5620,0.5130,0.3970,0.4500
middle,0.7510,0.3330,0.6580,0.4370,0.3060,0.3450,0.4980,0.5220,0.7000,0.7000,0.3920


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/06_h6_arc_semantics_audit/tables/w_tkr_raw.csv  (266 rows)


## 5. Close reading

Force-include low main-couple topics. In the cards, watch tertile tags on sentences:
rising end-conflict that is non-couple should not feed REFINED_RISING the same way
main-couple conflict does.

In [6]:
review_ids = rd.select_review_topics(
    h6,
    hyp=HYP,
    lex_idx=lex_idx,
    ctx_idx=ctx_idx,
    adj_idx=adj_idx,
    force_ids=low_mc["topic_id"].tolist(),
    per_code=3,
    seed=42,
)
packs = rd.show_review_set(
    cfg,
    h6,
    review_ids,
    hyp=HYP,
    lex_idx=lex_idx,
    ctx_idx=ctx_idx,
    adj_idx=adj_idx,
    code_col=CODE_COL,
    max_sentences=10,
)
ctx.save_markdown(
    rd.render_review_markdown(packs, title="H6 arc semantics — close-reading pack"),
    "close_reading_pack",
)
ctx.save_table(
    h6[
        [
            "topic_id",
            "current_topic_label",
            "current_taxonomy_id",
            "current_taxonomy_name",
            "arc_role",
            "code_norm",
            "main_couple_prob",
            "non_couple_prob",
        ]
    ],
    "arc_topic_annotations",
)

Close-reading 54 of 54 topics (disagreements / mixed / manual / stratified sample).

  TOPIC 6 — Whispered Reassurance
  Taxonomy : 4.6 — Emotional Safety, Reassurance & Caretaking
  Code     : ARC_0  (norm: ARC_0)
  Stage08 snippets:
      · i’ll make sure of it,” he whispered.
      · you’ll be fine,” he whispered. “
      · quiet, subdued, she nodded and then, “you’ll be safe.”
  Novel sentences:
    · [BOOK_001, CELL_D, tertile=end, p=0.62]
        Ian stared right back and asked slowly, clearly, enunciating every word, “Who asked
        you to?”
    · [BOOK_002, CELL_D, tertile=middle, p=0.74]
        Okay,’ she said, striving for a light tone in the heavy silence. ‘
  Pass A/B/C:
    A lexical: ARC_10
        All four keyword lists describe speech-act mechanics and vocal/emotional
        delivery cues (whispered, softly, tentatively, hesitation, drawled, solemnly,
        eagerly, confessed, mumbled) without anchoring to any specific relational event
        between a main coup

  TOPIC 157 — Swearing to Save Him From Himself
  Taxonomy : 4.5 — Reconciliation, Commitments & HEA
  Code     : ARC_7  (norm: ARC_7)
  Stage08 snippets:
      · and you’ll get it, i swear,” [person].
      · if [person] sees me like this, i'll die."
      · and i might be young but i guess that just makes me lucky… i do love him and whether
      · i have your help or not, i have to save him from himself… i know [person], i know
  Novel sentences:
    · [BOOK_006, CELL_B, tertile=middle, p=0.73]
        Sam couldn’t understand it, and that bothered him.
    · [BOOK_001, CELL_B, tertile=begin, p=0.82]
        What else had Sam witnessed?
    · [BOOK_006, CELL_B, tertile=begin, p=0.80]
        Sam noticed stuff like that.
  Pass A/B/C:
    A lexical: MIXED
        Main keywords (samantha, samara, sammi, name, optimism, jonathan) suggest
        character naming/identification with ambiguous relational valence — ARC_10.
        KeyBERT (begged, confessed, willing, tense, brief) points t

  TOPIC 285 — Confessing Years of Hatred
  Taxonomy : 3.2 — Negative Emotions & Distress
  Code     : ARC_2  (norm: ARC_2)
  Stage08 snippets:
      · i’ve hated him for years.
      · heath has done a wonderful job with him, but i—well, i’ve hated him for living
      · instead of you.
      · ever since this cunt came here, i’ve been, uh, less than balanced, i admit, and
      · remembering things i’ve tried to forget, reasons to hate, reasons to hate, and
  Novel sentences:
    · [BOOK_001, CELL_C, tertile=middle, p=0.57]
        Looking back now, I can see all the hatred I had in me, but at the time it just
        seemed like everyone was out to get me.
    · [BOOK_001, CELL_C, tertile=end, p=0.48]
        It had been a long time since his presence had evoked disdain.
    · [BOOK_002, CELL_B, tertile=middle, p=0.62]
        And I hated submitting to your licentious thoughts and advances!” “
  Pass A/B/C:
    A lexical: ARC_5
        Main keywords ('hate', 'hated', 'hates', 'hating

  TOPIC 319 — Confessing A Costly Mistake
  Taxonomy : 4.3 — Secrets, Misunderstandings & Hidden Information
  Code     : ARC_1  (norm: ARC_1)
  Stage08 snippets:
      · but you have a way of letting me know when you think i’ve made the wrong one.” “
      · i’ve made many mistakes and maybe i should have come back a year ago.
      · it was a mistake for which i’ve paid dearly.” “
  Novel sentences:
    · [BOOK_001, CELL_D, tertile=middle, p=0.42]
        They’re fundamental y flawed, I can’t—” “ ‘Flawed’ being the key word,” Josh points
        out.
  Pass A/B/C:
    A lexical: ARC_1
        Main keywords ('mistake', 'mistakes', 'flaw', 'error', 'errors', 'biggest',
        'terrible', 'made') strongly signal a misunderstanding or misjudgment frame —
        the vocabulary of recognizing a wrong belief or wrong action that drove a rift,
        which is the lexical signature of ARC_1 (misunderstanding). KeyBERT ('occurred',
        'thinks') and POS ('latest', 'bigger') are too spar

Later: REFINED_FALLING / REFINED_RISING deltas and RARC; EXTERNAL_PLOT_CONFLICT kept
outside the arc equation.

print("H6 audit review complete.")